# Modelagem Preditiva - Regressão (MonthlyIncome)

Neste notebook, desenvolvemos modelos de regressão para prever o rendimento mensal (`MonthlyIncome`) com base em características dos colaboradores. Seguiremos a arquitetura de Scikit-Learn Pipelines para encapsular todo o fluxo de engenharia de recursos e treino.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

In [ ]:
df = pd.read_csv("../data/raw/employee_data.csv")
df.head()

In [ ]:
# Divisão entre características (X) e alvo (y)
# Excluímos as colunas de variância zero identificadas na EDA (EmployeeCount, StandardHours, Over18)
X = df.drop(["MonthlyIncome", "EmployeeCount", "StandardHours", "Over18"], axis=1)
y = df["MonthlyIncome"]

In [ ]:
# Divisão do dataset em treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Identificação das colunas numéricas e categóricas
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numéricas:", list(numeric_features))
print("Categóricas:", list(categorical_features))

In [ ]:
# Criação do pré-processamento em pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# 1. Linear Regression (Baseline)
lr_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

lr_model.fit(X_train, y_train)
lr_y_pred = lr_model.predict(X_test)

print("Linear Regression R2:", r2_score(y_test, lr_y_pred))
print("Linear Regression RMSE:", root_mean_squared_error(y_test, lr_y_pred))
print("Linear Regression MAE:", mean_absolute_error(y_test, lr_y_pred))

In [ ]:
# 2. Random Forest Regressor
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=200, random_state=42))
    ]
)

rf_model.fit(X_train, y_train)
rf_y_pred = rf_model.predict(X_test)

print("Random Forest R2:", r2_score(y_test, rf_y_pred))
print("Random Forest RMSE:", root_mean_squared_error(y_test, rf_y_pred))
print("Random Forest MAE:", mean_absolute_error(y_test, rf_y_pred))

In [ ]:
# 3. Gradient Boosting Regressor
gb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", GradientBoostingRegressor(random_state=42))
    ]
)

gb_model.fit(X_train, y_train)
gb_y_pred = gb_model.predict(X_test)

print("Gradient Boosting R2:", r2_score(y_test, gb_y_pred))
print("Gradient Boosting RMSE:", root_mean_squared_error(y_test, gb_y_pred))
print("Gradient Boosting MAE:", mean_absolute_error(y_test, gb_y_pred))

In [ ]:
# 4. Otimização de Hiperparâmetros (GridSearchCV) no Gradient Boosting
param_grid = {
    "regressor__n_estimators": [100, 200],
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__max_depth": [3, 4]
}

grid_search = GridSearchCV(
    gb_model,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Melhores Hiperparâmetros:", grid_search.best_params_)

best_model = grid_search.best_estimator_
best_y_pred = best_model.predict(X_test)

print("Tuned Gradient Boosting R2:", r2_score(y_test, best_y_pred))
print("Tuned Gradient Boosting RMSE:", root_mean_squared_error(y_test, best_y_pred))
print("Tuned Gradient Boosting MAE:", mean_absolute_error(y_test, best_y_pred))

In [ ]:
# Comparação de todos os modelos avaliados
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting (Tuned)"],
    "R2 Score": [
        r2_score(y_test, lr_y_pred),
        r2_score(y_test, rf_y_pred),
        r2_score(y_test, best_y_pred)
    ],
    "RMSE": [
        root_mean_squared_error(y_test, lr_y_pred),
        root_mean_squared_error(y_test, rf_y_pred),
        root_mean_squared_error(y_test, best_y_pred)
    ],
    "MAE": [
        mean_absolute_error(y_test, lr_y_pred),
        mean_absolute_error(y_test, rf_y_pred),
        mean_absolute_error(y_test, best_y_pred)
    ]
})
results

## Comparação e Escolha do Modelo Final

Neste experimento, comparamos a regressão linear (baseline) com dois algoritmos baseados em árvores de decisão: Random Forest Regressor e Gradient Boosting Regressor.

A Regressão Linear apresentou excelente capacidade de predição com $R^2 \approx 0.9443$, o que reflete a forte relação linear de variáveis como `JobLevel` no rendimento.

O modelo Random Forest melhorou os resultados ($R^2 \approx 0.9505$), demonstrando que a modelagem de interações não lineares entre cargo, tempo de empresa e idade favorece a precisão.

Por fim, o **Gradient Boosting Regressor otimizado via GridSearchCV** obteve o melhor desempenho global de todos, alcançando $R^2 \approx 0.9523$ e reduzindo o RMSE para aproximadamente **1005.90** no conjunto de teste.

Portanto, o Gradient Boosting Regressor com os hiperparâmetros otimizados foi selecionado como o modelo final.

In [ ]:
import pickle

with open("../models/pipeline_regression.pkl", "wb") as file:
    pickle.dump(best_model, file)

In [ ]:
# Carregamento e validação rápida do pipeline exportado
with open("../models/pipeline_regression.pkl", "rb") as file:
    loaded_model = pickle.load(file)

test_predictions = loaded_model.predict(X_test)
print(test_predictions[:10])